# RoBERTa Fine-tuning (Kaggle Version)
## Domain-Adapted Fake News Detection

Fine-tune pretrained RoBERTa-base trên dữ liệu tin tức để phát hiện tin giả.


## 0. Kaggle Environment Setup

Cell này tự động:
- Fix numpy version conflict
- Detect dataset trong `/kaggle/input/`
- Copy source code (`src/`) vào `/kaggle/working/`
- Override config paths cho Kaggle
- Cài đặt dependencies thiếu


In [ ]:
# ============================================================
# Cell 0a: Cài đặt accelerate (Bỏ netcal để tránh numpy conflict)
# ============================================================
# netcal sẽ downgrade numpy → break scipy/sklearn khi Run All
# Code không dùng netcal, chỉ cần accelerate

!pip install -q accelerate

print("accelerate installed")
print("Using Kaggle's default: numpy, scipy, scikit-learn, torch, transformers")


In [ ]:
# ============================================================
# Cell 0b: Detect & Setup Kaggle Dataset
# ============================================================
import os, sys, glob, shutil
import numpy as np
import scipy

# Kiểm tra nhanh, nếu đúng version mới chạy tiếp
print(f"Môi trường sạch: Numpy {np.__version__}, Scipy {scipy.__version__}")

# --- Detect platform ---
IS_KAGGLE = os.path.exists('/kaggle')
assert IS_KAGGLE, "Notebook này được thiết kế cho Kaggle! Hãy upload lên Kaggle để chạy."
print("Đang chạy trên Kaggle")

# --- Tìm dataset trong /kaggle/input/ ---
INPUT_DIR = '/kaggle/input'
WORKING_DIR = '/kaggle/working'

# Liệt kê tất cả datasets đã add
datasets = [d for d in os.listdir(INPUT_DIR) if os.path.isdir(os.path.join(INPUT_DIR, d))]
print(f"\n Datasets đã thêm: {datasets}")

# --- Tìm thư mục src/ và data processed ---
SRC_FOUND = None
DATA_FOUND = None
ZIP_FOUND = None

for ds in datasets:
    ds_path = os.path.join(INPUT_DIR, ds)
    
    # Tìm src/ trực tiếp
    for root, dirs, files in os.walk(ds_path):
        if 'src' in dirs:
            src_candidate = os.path.join(root, 'src')
            if os.path.exists(os.path.join(src_candidate, 'config.py')):
                SRC_FOUND = src_candidate
        # Tìm processed data
        if 'processed' in dirs:
            proc_candidate = os.path.join(root, 'processed')
            if os.path.exists(os.path.join(proc_candidate, 'train.csv')):
                DATA_FOUND = proc_candidate
        # Tìm data/processed
        if 'data' in dirs:
            data_proc = os.path.join(root, 'data', 'processed')
            if os.path.exists(data_proc) and os.path.exists(os.path.join(data_proc, 'train.csv')):
                DATA_FOUND = data_proc
        # Depth limit
        if root.count(os.sep) - ds_path.count(os.sep) > 3:
            break
    
    # Tìm file zip
    for f in glob.glob(os.path.join(ds_path, '**/*.zip'), recursive=True):
        if 'roberta' in f.lower() or 'fake' in f.lower():
            ZIP_FOUND = f

# --- Nếu chưa tìm thấy, thử giải nén zip ---
if (SRC_FOUND is None or DATA_FOUND is None) and ZIP_FOUND:
    print(f"\n Tìm thấy file zip: {ZIP_FOUND}")
    print("   Đang giải nén...")
    !unzip -o "{ZIP_FOUND}" -d /kaggle/working/project/ -q
    print("   Giải nén thành công!")
    
    # Tìm lại sau khi giải nén
    for root, dirs, files in os.walk('/kaggle/working/project'):
        if 'src' in dirs and SRC_FOUND is None:
            src_candidate = os.path.join(root, 'src')
            if os.path.exists(os.path.join(src_candidate, 'config.py')):
                SRC_FOUND = src_candidate
        if 'processed' in dirs and DATA_FOUND is None:
            proc_candidate = os.path.join(root, 'processed')
            if os.path.exists(os.path.join(proc_candidate, 'train.csv')):
                DATA_FOUND = proc_candidate
        if 'data' in dirs and DATA_FOUND is None:
            data_proc = os.path.join(root, 'data', 'processed')
            if os.path.exists(data_proc) and os.path.exists(os.path.join(data_proc, 'train.csv')):
                DATA_FOUND = data_proc

print(f"\nKết quả tìm kiếm:")
print(f"   src/ folder: {SRC_FOUND or 'KHÔNG TÌM THẤY'}")
print(f"   processed/  : {DATA_FOUND or 'KHÔNG TÌM THẤY'}")

assert SRC_FOUND is not None, (
    "Không tìm thấy thư mục src/! "
    "Hãy upload toàn bộ project (bao gồm src/) lên Kaggle Dataset."
)
assert DATA_FOUND is not None, (
    "Không tìm thấy data processed (train.csv)! "
    "Hãy upload thư mục data/processed/ lên Kaggle Dataset."
)

In [ ]:
# ============================================================
# Cell 0c: Copy src/ sang /kaggle/working/ (vì /kaggle/input/ là read-only)
# ============================================================
WORK_SRC = os.path.join(WORKING_DIR, 'src')

if os.path.exists(WORK_SRC):
    shutil.rmtree(WORK_SRC)

shutil.copytree(SRC_FOUND, WORK_SRC)
print(f"Copied src/ → {WORK_SRC}")

# Verify
src_files = os.listdir(WORK_SRC)
print(f"  Files: {[f for f in src_files if f.endswith('.py')]}")

# Thêm working dir vào sys.path
if WORKING_DIR not in sys.path:
    sys.path.insert(0, WORKING_DIR)
    print(f"Added {WORKING_DIR} to sys.path")

In [ ]:
# ============================================================
# Cell 0d: Override config paths cho Kaggle
# ============================================================
import importlib.util

# Import TRỰC TIẾP config.py (bypass __init__.py để tránh chain import lỗi)
spec = importlib.util.spec_from_file_location('src.config', os.path.join(WORK_SRC, 'config.py'))
config = importlib.util.module_from_spec(spec)
sys.modules['src.config'] = config  # Register để các module khác dùng được
spec.loader.exec_module(config)

# Override paths cho Kaggle
config.BASE_DIR = WORKING_DIR
config.DATA_DIR = os.path.dirname(DATA_FOUND)  # parent of 'processed'
config.PROCESSED_DIR = DATA_FOUND
config.TRAIN_FILE = os.path.join(DATA_FOUND, 'train.csv')
config.VAL_FILE = os.path.join(DATA_FOUND, 'val.csv')
config.TEST_FILE = os.path.join(DATA_FOUND, 'test.csv')

# Output dirs (writable)
config.MODEL_SAVE_DIR = os.path.join(WORKING_DIR, 'models')
config.BASELINE_MODEL_DIR = os.path.join(config.MODEL_SAVE_DIR, 'baseline')
config.ROBERTA_MODEL_DIR = os.path.join(config.MODEL_SAVE_DIR, 'roberta')
config.RESULTS_DIR = os.path.join(WORKING_DIR, 'results')

# Tạo output dirs
os.makedirs(config.MODEL_SAVE_DIR, exist_ok=True)
os.makedirs(config.RESULTS_DIR, exist_ok=True)

# Verify
print("[OK] Config paths đã được override cho Kaggle:")
print(f"  TRAIN_FILE:       {config.TRAIN_FILE}  → exists: {os.path.exists(config.TRAIN_FILE)}")
print(f"  VAL_FILE:         {config.VAL_FILE}  → exists: {os.path.exists(config.VAL_FILE)}")
print(f"  TEST_FILE:        {config.TEST_FILE}  → exists: {os.path.exists(config.TEST_FILE)}")
print(f"  ROBERTA_MODEL_DIR: {config.ROBERTA_MODEL_DIR}")
print(f"  RESULTS_DIR:       {config.RESULTS_DIR}")

In [ ]:
# ============================================================
# Cell 0e: Import libraries & Setup
# ============================================================
import torch
from torch.utils.data import DataLoader

from src.config import *
from src.utils import set_seed, get_device, print_gpu_memory
from src.dataset import prepare_datasets
from src.model import load_model_and_tokenizer, save_model, get_model_summary, freeze_layers
from src.trainer import Trainer
from src.evaluation import (
    evaluate_model, print_evaluation_report,
    plot_confusion_matrix, plot_training_history, plot_roc_curve
)

set_seed(SEED)
device = get_device()
print_gpu_memory()

---
## 1. Load Model & Tokenizer


In [ ]:
model, tokenizer = load_model_and_tokenizer(
    model_name=MODEL_NAME,
    num_labels=NUM_LABELS,
    dropout_rate=DROPOUT_RATE
)
get_model_summary(model)

## 2. Prepare Datasets


In [ ]:
train_dataset, val_dataset, test_dataset = prepare_datasets(
    train_file=TRAIN_FILE,
    val_file=VAL_FILE,
    test_file=TEST_FILE,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

# Kaggle: dùng num_workers=0 để tránh lỗi multiprocessing trong notebook
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=0, pin_memory=True)

print(f'\nBatch size: {BATCH_SIZE}')
print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')
print(f'Test batches:  {len(test_loader)}')

## 3. (Optional) Freeze Bottom Layers

Đóng băng các layers đầu để giữ kiến thức tổng quát, chỉ fine-tune layers cuối.


In [ ]:
# Uncomment to freeze bottom 6 layers (of 12)
# model = freeze_layers(model, num_layers_to_freeze=6)
# get_model_summary(model)

## 4. Training


In [ ]:
trainer = Trainer(
    model=model,
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    device=device,
    learning_rate=LEARNING_RATE,
    num_epochs=NUM_EPOCHS,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    early_stopping_patience=EARLY_STOPPING_PATIENCE
)

history = trainer.train()

# Lưu model ngay sau khi train xong để tránh mất dữ liệu lần nữa
save_model(model, tokenizer, ROBERTA_MODEL_DIR)
print(f'\nModel đã được lưu tại: {ROBERTA_MODEL_DIR}')

## 5. Training History Visualization


In [ ]:
plot_training_history(history, save_path=os.path.join(RESULTS_DIR, 'roberta_training_history.png'))

## 6. Evaluate on Test Set


In [ ]:
predictions, true_labels, all_probs, metrics = evaluate_model(model, test_loader, device)

print_evaluation_report(predictions, true_labels)

# Save metrics
from src.utils import save_metrics
roberta_results = {
    'model': 'RoBERTa (fine-tuned)',
    **{k: float(v) for k, v in metrics.items()}
}
save_metrics(roberta_results, os.path.join(RESULTS_DIR, 'roberta_results.json'))

In [ ]:
# Confusion Matrix
plot_confusion_matrix(predictions, true_labels,
    save_path=os.path.join(RESULTS_DIR, 'roberta_confusion_matrix.png'),
    title='RoBERTa — Confusion Matrix')

In [ ]:
# ROC Curve
roc_auc = plot_roc_curve(true_labels, all_probs,
    save_path=os.path.join(RESULTS_DIR, 'roberta_roc_curve.png'),
    title='RoBERTa - ROC Curve')
print(f'ROC AUC: {roc_auc:.4f}')

## 7. Save Trained Model


In [ ]:
save_model(model, tokenizer, ROBERTA_MODEL_DIR)
print(f'\n[OK] Model saved to: {ROBERTA_MODEL_DIR}')

## 8. Quick Sanity Check


In [ ]:
from src.evaluation import predict_single

test_texts = [
    "The president announced new economic policies today during the press conference.",
    "BREAKING: Scientists discover that the moon is actually made of cheese!",
]

for text in test_texts:
    result = predict_single(model, tokenizer, text, device)
    print(f'\nText: {text[:80]}...')
    print(f'  Prediction: {result["prediction"]} ({result["confidence"]:.1%})')
    print(f'  Probs: real={result["probabilities"]["real"]:.3f}, fake={result["probabilities"]["fake"]:.3f}')

print('\n[OK] RoBERTa fine-tuning complete!')

## 9. Download Results từ Kaggle

Sau khi training xong, các kết quả sẽ nằm tại:
- **Model**: `/kaggle/working/models/roberta/`
- **Results**: `/kaggle/working/results/`

Để download, vào tab **Output** ở sidebar phải của Kaggle Notebook.

Hoặc chạy cell dưới để nén output thành 1 file zip tiện download:


In [ ]:
# ============================================================
# Cell 9: Đóng gói & Download (Bản chống lỗi 404)
# ============================================================
import zipfile
import os
from IPython.display import FileLink

output_zip = '/kaggle/working/roberta_output.zip'

# Xóa file cũ nếu có
if os.path.exists(output_zip):
    os.remove(output_zip)

print("[INFO] Đang nén kết quả...")
with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Nén Model và Results
    for folder in [ROBERTA_MODEL_DIR, RESULTS_DIR]:
        if os.path.exists(folder):
            for root, dirs, files in os.walk(folder):
                for file in files:
                    filepath = os.path.join(root, file)
                    arcname = os.path.relpath(filepath, WORKING_DIR)
                    zipf.write(filepath, arcname)

if os.path.exists(output_zip):
    print(f'Đã nén xong! Dung lượng: {os.path.getsize(output_zip) / 1e6:.1f} MB')
    print("\n[DOWNLOAD TẠI ĐÂY]:")
    display(FileLink('roberta_output.zip')) # Link tải trực tiếp
else:
    print("Không thể tạo file zip!")